In [1]:
import glob
import pandas as pd
import re
from bs4 import BeautifulSoup
import markdownify as md

from typing import List
from pydantic import BaseModel, Field
import instructor
from openai import OpenAI
from tqdm import tqdm
from zipfile import ZipFile

In [2]:
# 1. Define schema for categorized keywords
class JobKeywords(BaseModel):
    job_title: str = Field(description="Title of the job e.g., Assistant Professor of Economics")
    university: str = Field(description="Name of the university or institution offering the job, e.g., Salisbury University, or University of California, Berkeley")
    location: str = Field(description="Location of the job, e.g., Salisbury, MD, or Berkeley, CA")
    review_start: str = Field(description="Date from which they will begin reviewing applications (may be 'none'), e.g., September 11, 2026, or 2023-09-30")
    deadline: str = Field(description="Deadline for applying to the job (may be 'none'), e.g., September 11, 2026, or 2023-09-30")
    responsibilities: List[str] = Field(
        description="Key responsibilities or duties of the job (e.g., Undergraduate Teaching, Research, Publications, Service)"
    )
    preferred_qualifications: List[str] = Field(
        description="Preferred qualifications or experiences (e.g., Experience with Python, Strong communication skills)"
    )
    required_skills: List[str] = Field(
        description="Required technical skills, academic disciplines, or methodologies (e.g., Stata, Econometrics, Data Science)"
    )
    visa_sponsorship: str = Field(description="Visa sponsorship status, may or may not be mentioned. (e.g., Is eligible for H1B visa sponsorship, Not eligible, Not Mentioned)")

In [3]:
class RequiredDocuments(BaseModel):
    documents: List[str] = Field(description="List of required documents (e.g., Resume, Cover Letter, Transcripts, Teaching Statement, Research Statement, Diversity Statement, Teaching Portfolio, Letters of Recommendation)")

In [4]:
# 2. Initialize Instructor with local Ollama
client = instructor.from_openai(
    OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"),
    mode=instructor.Mode.JSON
)

/opt/homebrew/Caskroom/miniconda/base/envs/scraping/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
# 4. Extract keywords
def extract_from_post(job_description):
    extracted = client.chat.completions.create(
        model="qwen2.5:3b",
        response_model=JobKeywords,
        messages=[
            {
                "role": "system",
                "content": "Extract key information accurately from the provided job description text."
            },
            {
                "role": "user",
                "content": job_description
            }
        ]
    )
    return extracted.model_dump()

In [6]:
# 4. Extract keywords
def extract_requirements_from_post(job_description):
    extracted = client.chat.completions.create(
        model="qwen2.5:3b",
        response_model=RequiredDocuments,
        messages=[
            {
                "role": "system",
                "content": "Extract required documents accurately from the provided job description text."
            },
            {
                "role": "user",
                "content": job_description
            }
        ]
    )
    return extracted.model_dump()

In [27]:
# Open the ZIP archive
with ZipFile('data/html_cache/cv1.zip', 'r') as zf:
    # Loop through all files in the ZIP folder
    for file_name in zf.namelist():
        # Optional: Filter for specific file types (e.g., text files)
        if file_name.endswith('.html'):
            print(f"--- Reading: {file_name} ---")

            # Open the specific file inside the ZIP
            with zf.open(file_name, 'r') as f:
                for line in f:
                    # Decode bytes to string and print
                    print(line.decode('utf-8').strip())

--- Reading: cv/205.html ---
<!DOCTYPE html><html lang="en-US" class="js"><head>

<title>Research Assistant - Tepper School of Business job with Carnegie Mellon University | 37989709</title>




KeyboardInterrupt: 

In [9]:
files = glob.glob("data/html_cache/hedj/*.html")

In [10]:
attribs = ['job-header', 'jobLocation', 'jobAttrib', 'jobDesc']
info = []
for file in files:
    record = {"file": file, "text": "", "link": ""}
    with open(file, "r", encoding="utf-8") as f:
        html_content = f.read()
    soup = BeautifulSoup(html_content, "html.parser")
    for attrib_id in attribs:
        attrib_div = soup.find(id=attrib_id)
        if attrib_div:
            attrib = md.markdownify(str(attrib_div))
            record["text"] += attrib

    link_div = soup.find(id="js-applyurl", recursive=True)
    if link_div:
        # Get 'href' or fall back to custom attributes like 'data-orig-href'
        href = link_div.get("data-orig-href")
        record["link"] = href

    # Pattern handles arbitrary whitespace/tabs and captures the date string
    pattern = r'"datePosted"\s*:\s*"([^"]+)"'

    match = re.search(pattern, html_content)
    if match:
        date_posted = match.group(1)
        record["date_posted"] = date_posted

    info.append(record)

KeyboardInterrupt: 

In [41]:
df = pd.DataFrame(info)

In [ ]:
keywords = []

In [37]:
correct_idx = [r['idx'] for r in keywords]
failed_idx = []

In [38]:
for idx, row in tqdm(df.iterrows()):
    if idx not in correct_idx:
        try:
            kw = extract_from_post(row["text"])
            kw['idx'] = idx
            keywords.append(kw)
        except Exception as e:
            print(f"Failed to process post at index {idx}: {e}")
            failed_idx.append(idx)
    else:
        print("Skipped post at index", idx)


0it [00:00, ?it/s]

Skipped post at index 0
Skipped post at index 1
Skipped post at index 2
Skipped post at index 3
Skipped post at index 4
Skipped post at index 5
Skipped post at index 6
Skipped post at index 7
Skipped post at index 8
Skipped post at index 9
Skipped post at index 10
Skipped post at index 11
Skipped post at index 12
Skipped post at index 13
Skipped post at index 14
Skipped post at index 15
Skipped post at index 16
Skipped post at index 17
Skipped post at index 18
Skipped post at index 19
Skipped post at index 20
Skipped post at index 21
Skipped post at index 22
Skipped post at index 23
Skipped post at index 24
Skipped post at index 25
Skipped post at index 26
Skipped post at index 27
Skipped post at index 28
Skipped post at index 29
Skipped post at index 30
Skipped post at index 31
Skipped post at index 32
Skipped post at index 33
Skipped post at index 34
Skipped post at index 35
Skipped post at index 36
Skipped post at index 37
Skipped post at index 38
Skipped post at index 39
Skipped po

101it [00:16,  6.18it/s]

Skipped post at index 101
Skipped post at index 102
Skipped post at index 103
Skipped post at index 104
Skipped post at index 105
Skipped post at index 106
Skipped post at index 107
Skipped post at index 108
Skipped post at index 109
Skipped post at index 110
Skipped post at index 111
Skipped post at index 112
Skipped post at index 113
Skipped post at index 114
Skipped post at index 115
Skipped post at index 116
Skipped post at index 117
Skipped post at index 118
Skipped post at index 119
Skipped post at index 120
Skipped post at index 121
Skipped post at index 122
Skipped post at index 123
Skipped post at index 124
Skipped post at index 125
Skipped post at index 126
Skipped post at index 127
Skipped post at index 128
Skipped post at index 129
Skipped post at index 130
Skipped post at index 131
Skipped post at index 132
Skipped post at index 133
Skipped post at index 134
Skipped post at index 135
Skipped post at index 136
Skipped post at index 137
Skipped post at index 138
Skipped post

245it [00:34,  7.06it/s]

Skipped post at index 217
Skipped post at index 218
Skipped post at index 219
Skipped post at index 220
Skipped post at index 221
Skipped post at index 222
Skipped post at index 223
Skipped post at index 224
Skipped post at index 225
Skipped post at index 226
Skipped post at index 227
Skipped post at index 228
Skipped post at index 229
Skipped post at index 230
Skipped post at index 231
Skipped post at index 232
Skipped post at index 233
Skipped post at index 234
Skipped post at index 235
Skipped post at index 236
Skipped post at index 237
Skipped post at index 238
Skipped post at index 239
Skipped post at index 240
Skipped post at index 241
Skipped post at index 242
Skipped post at index 243
Skipped post at index 244


In [19]:
keywords

[{'job_title': 'Assistant Professor of Economics',
  'university': 'Salisbury University',
  'location': 'Salisbury, MD',
  'review_start': '2026-09-11',
  'deadline': '2026-09-11',
  'responsibilities': ['Undergraduate Teaching',
   'Research',
   'Publications',
   'Service'],
  'preferred_qualifications': ['Evidence of teaching effectiveness at the undergraduate level',
   "A research pipeline appropriate to a candidate's career stage, with publications or working papers in progress",
   'Interest in developing applied, policy-relevant electives that engage students',
   'Ability to work with data and modern econometric or computational tools'],
  'required_skills': ['Ph.D. in Economics',
   'Demonstrated potential for high-quality research in the relevant field',
   'Demonstrated commitment to excellence in undergraduate teaching'],
  'visa_sponsorship': 'Not eligible',
  'idx': 0},
 {'job_title': 'Postdoctoral Fellow in Economic Science, Fall 2027',
  'university': 'Chapman Univer

In [42]:
df_details = pd.DataFrame(keywords)

In [43]:
df_details

,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx
0,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"[Undergraduate Teaching, Research, Publication...",[Evidence of teaching effectiveness at the und...,"[Ph.D. in Economics, Demonstrated potential fo...",Not eligible,0
1,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"[Design, program and run experiments using hum...",[],"[PhD in Economics, any field, with an emphasis...",Not eligible,1
2,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"[Teaching, Research, Service]","[Experience in critical political economy, Tra...",[Strong teaching excellence],Not eligible,2
3,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,,"[Undergraduate Teaching, Co-teaching in areas ...",[],[Advanced degree or comparable experience in a...,Not eligible,3
4,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",,,"[Undergraduate Teaching, Research, Publication...",[Experience with relevant technologies and met...,"[Research in emerging academic disciplines, Te...",Is eligible for H1B visa sponsorship,4
...,...,...,...,...,...,...,...,...,...,...
329,Professor of the Practice of Global Economic D...,Boston University,"Boston, MA",none,2026-09-21,"[Teach courses in their areas of expertise, Ad...",[Experience with international economic develo...,[Being an international economist and distingu...,Not Mentioned,73
330,"Assistant Professor, Economics (Tenure-Track)",Minot State University,"Minot, ND",none,none,"[Undergraduate Teaching, Research, Service, Re...","[Experience with Python, Strong communication ...","[Stata, Econometrics, Data Science]",Not Mentioned,74
331,2026-27AY Quarterly Lecturer - Economics (Pool),Santa Clara University,"Santa Clara, CA",none,none,[Fulfilling all duties associated with these c...,[Undergraduate teaching experience],"[Experience with Economics, Ph. D. in Economics]",Not Mentioned,75
332,Business and Technology Adjunct Opportunities,University of Wisconsin - Parkside,"Kenosha, WI",none,none,"[* Accounting, * Data Analytics, * Finance, * ...",[],[],Not Mentioned,100


In [44]:
merged = pd.merge(df, df_details, left_index=True, right_on='idx')
merged

,file,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx
0,data/html_cache/hedj/205.html,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"[Undergraduate Teaching, Research, Publication...",[Evidence of teaching effectiveness at the und...,"[Ph.D. in Economics, Demonstrated potential fo...",Not eligible,0
1,data/html_cache/hedj/74.html,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"[Design, program and run experiments using hum...",[],"[PhD in Economics, any field, with an emphasis...",Not eligible,1
2,data/html_cache/hedj/23.html,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"[Teaching, Research, Service]","[Experience in critical political economy, Tra...",[Strong teaching excellence],Not eligible,2
3,data/html_cache/hedj/194.html,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,,"[Undergraduate Teaching, Co-teaching in areas ...",[],[Advanced degree or comparable experience in a...,Not eligible,3
4,data/html_cache/hedj/244.html,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",,,"[Undergraduate Teaching, Research, Publication...",[Experience with relevant technologies and met...,"[Research in emerging academic disciplines, Te...",Is eligible for H1B visa sponsorship,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,data/html_cache/hedj/139.html,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,[Teach undergraduate and graduate courses in t...,[],[Ph.D. in Economics or closely related field b...,Not Mentioned,240
239,data/html_cache/hedj/142.html,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"[Teaching in large-lecture sections, Teaching ...",[Experience with teaching large lecture format...,"[Ph. D. in Economics, Prior college/university...",Not Mentioned,241
240,data/html_cache/hedj/115.html,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",,Open Until Filled,"[carry out their own research, participate in ...","[strong academic record, trajectory to contrib...","[doctorate in May 2025 or later, engage with t...",Not Mentioned,242
241,data/html_cache/hedj/103.html,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.287,TTU Post Doctoral Research Associate,Texas Tech University,"Lubbock, TX",Not mentioned,Not mentioned,[Performs specialized Post Doctoral work in th...,[Demonstrated track record of published resear...,"[PhD in area of project specialization, Knowle...",Not Mentioned,243


In [45]:
merged_uniq = merged.drop_duplicates(subset=['file'], keep='last')
merged_uniq

,file,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx
0,data/html_cache/hedj/205.html,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"[Undergraduate Teaching, Research, Publication...",[Evidence of teaching effectiveness at the und...,"[Ph.D. in Economics, Demonstrated potential fo...",Not eligible,0
1,data/html_cache/hedj/74.html,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"[Design, program and run experiments using hum...",[],"[PhD in Economics, any field, with an emphasis...",Not eligible,1
2,data/html_cache/hedj/23.html,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"[Teaching, Research, Service]","[Experience in critical political economy, Tra...",[Strong teaching excellence],Not eligible,2
3,data/html_cache/hedj/194.html,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,,"[Undergraduate Teaching, Co-teaching in areas ...",[],[Advanced degree or comparable experience in a...,Not eligible,3
4,data/html_cache/hedj/244.html,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",,,"[Undergraduate Teaching, Research, Publication...",[Experience with relevant technologies and met...,"[Research in emerging academic disciplines, Te...",Is eligible for H1B visa sponsorship,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,data/html_cache/hedj/139.html,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,[Teach undergraduate and graduate courses in t...,[],[Ph.D. in Economics or closely related field b...,Not Mentioned,240
239,data/html_cache/hedj/142.html,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"[Teaching in large-lecture sections, Teaching ...",[Experience with teaching large lecture format...,"[Ph. D. in Economics, Prior college/university...",Not Mentioned,241
240,data/html_cache/hedj/115.html,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",,Open Until Filled,"[carry out their own research, participate in ...","[strong academic record, trajectory to contrib...","[doctorate in May 2025 or later, engage with t...",Not Mentioned,242
241,data/html_cache/hedj/103.html,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.287,TTU Post Doctoral Research Associate,Texas Tech University,"Lubbock, TX",Not mentioned,Not mentioned,[Performs specialized Post Doctoral work in th...,[Demonstrated track record of published resear...,"[PhD in area of project specialization, Knowle...",Not Mentioned,243


In [47]:
merged_uniq.to_csv(
    "data/output.csv",
    index=False,
    lineterminator="\r\n",  # Ensures standard carriage return + line feed for multi-line cells
    encoding="utf-8",
)

In [8]:
merged_uniq = pd.read_csv("data/output.csv")

In [9]:
keywords = []

In [13]:
correct_idx = [r['idx'] for r in keywords]
failed_idx = []

In [14]:
for idx, row in tqdm(merged_uniq.iterrows()):
    if idx not in correct_idx:
        try:
            kw = extract_requirements_from_post(row["text"])
            kw['idx'] = idx
            keywords.append(kw)
        except Exception as e:
            print(f"Failed to process post at index {idx}: {e}")
            failed_idx.append(idx)
    else:
        print("Skipped post at index", idx)

0it [00:00, ?it/s]

Skipped post at index 0
Skipped post at index 1
Skipped post at index 2
Skipped post at index 3
Skipped post at index 4
Skipped post at index 5
Skipped post at index 6
Skipped post at index 7
Skipped post at index 8
Skipped post at index 9
Skipped post at index 10
Skipped post at index 11
Skipped post at index 12
Skipped post at index 13
Skipped post at index 14
Skipped post at index 15
Skipped post at index 16
Skipped post at index 17
Skipped post at index 18
Skipped post at index 19
Skipped post at index 20
Skipped post at index 21
Skipped post at index 22
Skipped post at index 23
Skipped post at index 24
Skipped post at index 25
Skipped post at index 26
Skipped post at index 27
Skipped post at index 28
Skipped post at index 29
Skipped post at index 30
Skipped post at index 31
Skipped post at index 32
Skipped post at index 33
Skipped post at index 34
Skipped post at index 35
Skipped post at index 36
Skipped post at index 37
Skipped post at index 38
Skipped post at index 39
Skipped po

127it [00:15,  8.11it/s]

Skipped post at index 127
Skipped post at index 128
Skipped post at index 129
Skipped post at index 130
Skipped post at index 131
Skipped post at index 132
Skipped post at index 133
Skipped post at index 134
Skipped post at index 135
Skipped post at index 136
Skipped post at index 137


139it [00:28,  4.31it/s]

Skipped post at index 139
Skipped post at index 140
Skipped post at index 141
Skipped post at index 142
Skipped post at index 143
Skipped post at index 144
Skipped post at index 145
Skipped post at index 146
Skipped post at index 147
Skipped post at index 148
Skipped post at index 149
Skipped post at index 150
Skipped post at index 151
Skipped post at index 152
Skipped post at index 153
Skipped post at index 154
Skipped post at index 155
Skipped post at index 156
Skipped post at index 157
Skipped post at index 158
Skipped post at index 159
Skipped post at index 160
Skipped post at index 161
Skipped post at index 162
Skipped post at index 163
Skipped post at index 164
Skipped post at index 165
Skipped post at index 166
Skipped post at index 167
Skipped post at index 168
Skipped post at index 169
Skipped post at index 170
Skipped post at index 171
Skipped post at index 172
Skipped post at index 173
Skipped post at index 174
Skipped post at index 175
Skipped post at index 176


178it [00:56,  2.47it/s]

Skipped post at index 178
Skipped post at index 179
Skipped post at index 180
Skipped post at index 181
Skipped post at index 182
Skipped post at index 183
Skipped post at index 184
Skipped post at index 185
Skipped post at index 186
Skipped post at index 187
Skipped post at index 188
Skipped post at index 189
Skipped post at index 190
Skipped post at index 191
Skipped post at index 192
Skipped post at index 193
Skipped post at index 194
Skipped post at index 195
Skipped post at index 196
Skipped post at index 197
Skipped post at index 198
Skipped post at index 199
Skipped post at index 200
Skipped post at index 201
Skipped post at index 202
Skipped post at index 203
Skipped post at index 204
Skipped post at index 205
Skipped post at index 206
Skipped post at index 207
Skipped post at index 208
Skipped post at index 209
Skipped post at index 210
Skipped post at index 211
Skipped post at index 212
Skipped post at index 213
Skipped post at index 214
Skipped post at index 215
Skipped post

245it [01:10,  3.46it/s]

Skipped post at index 243
Skipped post at index 244


In [15]:
df_requirements = pd.DataFrame(keywords)
df_requirements

,documents,idx
0,"[Curriculum Vitae, Cover Letter, Job Market Pa...",0
1,"[Statement of interest, CV, Job Market Paper, ...",1
2,"[Resume, Cover Letter, Curriculum Vitae, State...",2
3,"[Curriculum Vitae, Cover Letter, Teaching Stat...",3
4,"[Resume, Cover Letter, Research Statement, Tea...",4
...,...,...
240,"[Resume, Cover Letter]",244
241,[Curriculum Vitae],126
242,"[Letter of Application, Curriculum Vitae, Stat...",138
243,"[Resume/CV, Cover Letter, Contact Information ...",177


In [16]:
merged2 = pd.merge(merged_uniq, df_requirements, left_on='idx', right_on='idx')
merged2

,file,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx,documents
0,data/html_cache/hedj/205.html,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"['Undergraduate Teaching', 'Research', 'Public...",['Evidence of teaching effectiveness at the un...,"['Ph.D. in Economics', 'Demonstrated potential...",Not eligible,0,"[Curriculum Vitae, Cover Letter, Job Market Pa..."
1,data/html_cache/hedj/74.html,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"['Design, program and run experiments using hu...",[],"['PhD in Economics, any field, with an emphasi...",Not eligible,1,"[Statement of interest, CV, Job Market Paper, ..."
2,data/html_cache/hedj/23.html,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"['Teaching', 'Research', 'Service']","['Experience in critical political economy', '...",['Strong teaching excellence'],Not eligible,2,"[Resume, Cover Letter, Curriculum Vitae, State..."
3,data/html_cache/hedj/194.html,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,NaN,"['Undergraduate Teaching', 'Co-teaching in are...",[],['Advanced degree or comparable experience in ...,Not eligible,3,"[Curriculum Vitae, Cover Letter, Teaching Stat..."
4,data/html_cache/hedj/244.html,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",NaN,NaN,"['Undergraduate Teaching', 'Research', 'Public...",['Experience with relevant technologies and me...,"['Research in emerging academic disciplines', ...",Is eligible for H1B visa sponsorship,4,"[Resume, Cover Letter, Research Statement, Tea..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,data/html_cache/hedj/139.html,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,"[""Teach undergraduate and graduate courses in ...",[],['Ph.D. in Economics or closely related field ...,Not Mentioned,240,"[Cover Letter, C.V., Research Statement, Teach..."
241,data/html_cache/hedj/142.html,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"['Teaching in large-lecture sections', ""Teachi...",['Experience with teaching large lecture forma...,"['Ph. D. in Economics', 'Prior college/univers...",Not Mentioned,241,"[Cover Letter, CV, Statement of Teaching Philo..."
242,data/html_cache/hedj/115.html,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",NaN,Open Until Filled,"['carry out their own research', 'participate ...","['strong academic record', 'trajectory to cont...","['doctorate in May 2025 or later', 'engage wit...",Not Mentioned,242,"[CV, Research statement (max 1,500 words), Wri..."
243,data/html_cache/hedj/103.html,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.2

In [19]:
# 1. Convert to datetime, turning malformed rows into NaT (Not a Time)
review_start = pd.to_datetime(merged2["review_start"], errors="coerce", format="mixed")
deadline = pd.to_datetime(merged2["deadline"], errors="coerce", format="mixed")

# 2. Format the valid dates to your fixed format (e.g., 'YYYY-MM-DD')
formatted_review_start = review_start.dt.strftime("%Y-%m-%d")
formatted_deadline = deadline.dt.strftime("%Y-%m-%d")

In [20]:
# 3. Fill the NaT gaps with the original text column
formatted_review_start = formatted_review_start.fillna(merged2["review_start"])
formatted_deadline = formatted_deadline.fillna(merged2["deadline"])

In [21]:
merged2["review_start"] = formatted_review_start
merged2["deadline"] = formatted_deadline
merged2

,file,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx,documents
0,data/html_cache/hedj/205.html,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"['Undergraduate Teaching', 'Research', 'Public...",['Evidence of teaching effectiveness at the un...,"['Ph.D. in Economics', 'Demonstrated potential...",Not eligible,0,"[Curriculum Vitae, Cover Letter, Job Market Pa..."
1,data/html_cache/hedj/74.html,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"['Design, program and run experiments using hu...",[],"['PhD in Economics, any field, with an emphasi...",Not eligible,1,"[Statement of interest, CV, Job Market Paper, ..."
2,data/html_cache/hedj/23.html,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"['Teaching', 'Research', 'Service']","['Experience in critical political economy', '...",['Strong teaching excellence'],Not eligible,2,"[Resume, Cover Letter, Curriculum Vitae, State..."
3,data/html_cache/hedj/194.html,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,NaN,"['Undergraduate Teaching', 'Co-teaching in are...",[],['Advanced degree or comparable experience in ...,Not eligible,3,"[Curriculum Vitae, Cover Letter, Teaching Stat..."
4,data/html_cache/hedj/244.html,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",NaN,NaN,"['Undergraduate Teaching', 'Research', 'Public...",['Experience with relevant technologies and me...,"['Research in emerging academic disciplines', ...",Is eligible for H1B visa sponsorship,4,"[Resume, Cover Letter, Research Statement, Tea..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,data/html_cache/hedj/139.html,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,"[""Teach undergraduate and graduate courses in ...",[],['Ph.D. in Economics or closely related field ...,Not Mentioned,240,"[Cover Letter, C.V., Research Statement, Teach..."
241,data/html_cache/hedj/142.html,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"['Teaching in large-lecture sections', ""Teachi...",['Experience with teaching large lecture forma...,"['Ph. D. in Economics', 'Prior college/univers...",Not Mentioned,241,"[Cover Letter, CV, Statement of Teaching Philo..."
242,data/html_cache/hedj/115.html,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",NaN,Open Until Filled,"['carry out their own research', 'participate ...","['strong academic record', 'trajectory to cont...","['doctorate in May 2025 or later', 'engage wit...",Not Mentioned,242,"[CV, Research statement (max 1,500 words), Wri..."
243,data/html_cache/hedj/103.html,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.2

In [26]:
merged2['status'] = '...'
merged2.to_csv(
    "data/hedjobs.csv",
    index=False,
    lineterminator="\r\n",  # Ensures standard carriage return + line feed for multi-line cells
    encoding="utf-8",
)

In [24]:
docs = merged2["documents"]
documents = set()
for ds in docs:
    # s = [str(i) for i in ds[1:-1].split(',')]
    documents.update(ds)
documents

{'2 publications or writing samples',
 '2 recommendation letters',
 '3 Reference Letters (contact information only)',
 "A cover letter addressing the position requirements listed above as well as the ways they might contribute to and further the College's Jesuit, undergraduate liberal arts college and its core commitment to creating a community where every individual feels empowered to contribute their talents and perspectives.",
 'A curriculum vitae',
 'A letter of application detailing your interest in the position and how you meet the required qualifications (max. 2 pages) (as one complete document).',
 'A statement of teaching philosophy',
 "A statement of teaching philosophy, including evidence of teaching effectiveness, supporting URI's strategic priorities of cultivating a diverse, inclusive, equitable, and accessible learning environment (max. 3 pages) (as one complete document).",
 'A statement of your teaching philosophy and interests, with particular emphasis on how you util

In [25]:
merged2['status'] = 'Open'


,file,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx,documents
0,data/html_cache/hedj/205.html,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"['Undergraduate Teaching', 'Research', 'Public...",['Evidence of teaching effectiveness at the un...,"['Ph.D. in Economics', 'Demonstrated potential...",Not eligible,0,"[Curriculum Vitae, Cover Letter, Job Market Pa..."
1,data/html_cache/hedj/74.html,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"['Design, program and run experiments using hu...",[],"['PhD in Economics, any field, with an emphasi...",Not eligible,1,"[Statement of interest, CV, Job Market Paper, ..."
2,data/html_cache/hedj/23.html,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"['Teaching', 'Research', 'Service']","['Experience in critical political economy', '...",['Strong teaching excellence'],Not eligible,2,"[Resume, Cover Letter, Curriculum Vitae, State..."
3,data/html_cache/hedj/194.html,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,NaN,"['Undergraduate Teaching', 'Co-teaching in are...",[],['Advanced degree or comparable experience in ...,Not eligible,3,"[Curriculum Vitae, Cover Letter, Teaching Stat..."
4,data/html_cache/hedj/244.html,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",NaN,NaN,"['Undergraduate Teaching', 'Research', 'Public...",['Experience with relevant technologies and me...,"['Research in emerging academic disciplines', ...",Is eligible for H1B visa sponsorship,4,"[Resume, Cover Letter, Research Statement, Tea..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,data/html_cache/hedj/139.html,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,"[""Teach undergraduate and graduate courses in ...",[],['Ph.D. in Economics or closely related field ...,Not Mentioned,240,"[Cover Letter, C.V., Research Statement, Teach..."
241,data/html_cache/hedj/142.html,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"['Teaching in large-lecture sections', ""Teachi...",['Experience with teaching large lecture forma...,"['Ph. D. in Economics', 'Prior college/univers...",Not Mentioned,241,"[Cover Letter, CV, Statement of Teaching Philo..."
242,data/html_cache/hedj/115.html,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",NaN,Open Until Filled,"['carry out their own research', 'participate ...","['strong academic record', 'trajectory to cont...","['doctorate in May 2025 or later', 'engage wit...",Not Mentioned,242,"[CV, Research statement (max 1,500 words), Wri..."
243,data/html_cache/hedj/103.html,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.2

In [4]:
import glob
import pandas as pd
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs

In [2]:
files = glob.glob("data/html_cache/hedj/*.html")

In [18]:
ids = []
mappings = {}
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        html_content = f.read()
    soup = BeautifulSoup(html_content, "lxml")

    # inputs = soup.find_all('input', {'data-jobcode': True})
    spans = soup.find_all('span', {'data-jc': True}, recursive=True)

    # 2. Extract the JobCode from each link
    print(f"File: {file}")
    if spans:
        id = spans[0]['data-jc']
        print(id)
        ids.append(id)
        mappings[file] = id

File: data/html_cache/hedj/205.html
179507024
File: data/html_cache/hedj/74.html
179568871
File: data/html_cache/hedj/23.html
179582487
File: data/html_cache/hedj/194.html
179521072
File: data/html_cache/hedj/244.html
177351010
File: data/html_cache/hedj/35.html
179580324
File: data/html_cache/hedj/182.html
179532598
File: data/html_cache/hedj/213.html
179488833
File: data/html_cache/hedj/62.html
179572493
File: data/html_cache/hedj/9.html
179587474
File: data/html_cache/hedj/141.html
179548005
File: data/html_cache/hedj/19.html
179583833
File: data/html_cache/hedj/116.html
179557582
File: data/html_cache/hedj/229.html
179316650
File: data/html_cache/hedj/100.html
179561688
File: data/html_cache/hedj/58.html
179574139
File: data/html_cache/hedj/157.html
179543480
File: data/html_cache/hedj/120.html
179556511
File: data/html_cache/hedj/78.html
179567015
File: data/html_cache/hedj/97.html
179562585
File: data/html_cache/hedj/209.html
179501121
File: data/html_cache/hedj/177.html
17953469

In [19]:
uniqids = set(ids)
print(len(uniqids), len(ids))

244 244


In [20]:
print(len(mappings), len(files))

244 245


In [22]:
from pathlib import Path

for file, id in mappings.items():
    print(f"{file} --> data/html_cache/hedj/{id}.html")
    # # Define the file paths
    # old_file = Path("documents/old_name.txt")
    # new_file = old_file.with_name("new_name.txt")
    #
    # # Rename the file
    # old_file.rename(new_file)

data/html_cache/hedj/205.html --> data/html_cache/hedj/179507024.html
data/html_cache/hedj/74.html --> data/html_cache/hedj/179568871.html
data/html_cache/hedj/23.html --> data/html_cache/hedj/179582487.html
data/html_cache/hedj/194.html --> data/html_cache/hedj/179521072.html
data/html_cache/hedj/244.html --> data/html_cache/hedj/177351010.html
data/html_cache/hedj/35.html --> data/html_cache/hedj/179580324.html
data/html_cache/hedj/182.html --> data/html_cache/hedj/179532598.html
data/html_cache/hedj/213.html --> data/html_cache/hedj/179488833.html
data/html_cache/hedj/62.html --> data/html_cache/hedj/179572493.html
data/html_cache/hedj/9.html --> data/html_cache/hedj/179587474.html
data/html_cache/hedj/141.html --> data/html_cache/hedj/179548005.html
data/html_cache/hedj/19.html --> data/html_cache/hedj/179583833.html
data/html_cache/hedj/116.html --> data/html_cache/hedj/179557582.html
data/html_cache/hedj/229.html --> data/html_cache/hedj/179316650.html
data/html_cache/hedj/100.ht

In [23]:
df = pd.read_csv("data/hedjobs.csv")

In [26]:
df['file'] = df['file'].replace(mappings)
df

,file,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx,documents,status
0,179507024,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"['Undergraduate Teaching', 'Research', 'Public...",['Evidence of teaching effectiveness at the un...,"['Ph.D. in Economics', 'Demonstrated potential...",Not eligible,0,"['Curriculum Vitae', 'Cover Letter', 'Job Mark...",...
1,179568871,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"['Design, program and run experiments using hu...",[],"['PhD in Economics, any field, with an emphasi...",Not eligible,1,"['Statement of interest', 'CV', 'Job Market Pa...",n
2,179582487,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"['Teaching', 'Research', 'Service']","['Experience in critical political economy', '...",['Strong teaching excellence'],Not eligible,2,"['Resume', 'Cover Letter', 'Curriculum Vitae',...",n
3,179521072,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,NaN,"['Undergraduate Teaching', 'Co-teaching in are...",[],['Advanced degree or comparable experience in ...,Not eligible,3,"['Curriculum Vitae', 'Cover Letter', 'Teaching...",n
4,177351010,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",NaN,NaN,"['Undergraduate Teaching', 'Research', 'Public...",['Experience with relevant technologies and me...,"['Research in emerging academic disciplines', ...",Is eligible for H1B visa sponsorship,4,"['Resume', 'Cover Letter', 'Research Statement...",n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,179548639,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,"[""Teach undergraduate and graduate courses in ...",[],['Ph.D. in Economics or closely related field ...,Not Mentioned,240,"['Cover Letter', 'C.V.', 'Research Statement',...",y
241,179547688,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"['Teaching in large-lecture sections', ""Teachi...",['Experience with teaching large lecture forma...,"['Ph. D. in Economics', 'Prior college/univers...",Not Mentioned,241,"['Cover Letter', 'CV', 'Statement of Teaching ...",m
242,179557793,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",NaN,Open Until Filled,"['carry out their own research', 'participate ...","['strong academic record', 'trajectory to cont...","['doctorate in May 2025 or later', 'engage wit...",Not Mentioned,242,"['CV', 'Research statement (max 1,500 words)',...",n
243,179561620,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.287,TTU Post Doctoral Research Associate,Texas Tech University,"Lubbock, TX",Not mentioned,Not mentioned,['Performs specialized Post Doctoral work in 

In [27]:
df = df.rename(columns={'file': 'id'})
df

,id,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,idx,documents,status
0,179507024,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"['Undergraduate Teaching', 'Research', 'Public...",['Evidence of teaching effectiveness at the un...,"['Ph.D. in Economics', 'Demonstrated potential...",Not eligible,0,"['Curriculum Vitae', 'Cover Letter', 'Job Mark...",...
1,179568871,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"['Design, program and run experiments using hu...",[],"['PhD in Economics, any field, with an emphasi...",Not eligible,1,"['Statement of interest', 'CV', 'Job Market Pa...",n
2,179582487,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"['Teaching', 'Research', 'Service']","['Experience in critical political economy', '...",['Strong teaching excellence'],Not eligible,2,"['Resume', 'Cover Letter', 'Curriculum Vitae',...",n
3,179521072,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,NaN,"['Undergraduate Teaching', 'Co-teaching in are...",[],['Advanced degree or comparable experience in ...,Not eligible,3,"['Curriculum Vitae', 'Cover Letter', 'Teaching...",n
4,177351010,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",NaN,NaN,"['Undergraduate Teaching', 'Research', 'Public...",['Experience with relevant technologies and me...,"['Research in emerging academic disciplines', ...",Is eligible for H1B visa sponsorship,4,"['Resume', 'Cover Letter', 'Research Statement...",n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,179548639,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,"[""Teach undergraduate and graduate courses in ...",[],['Ph.D. in Economics or closely related field ...,Not Mentioned,240,"['Cover Letter', 'C.V.', 'Research Statement',...",y
241,179547688,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"['Teaching in large-lecture sections', ""Teachi...",['Experience with teaching large lecture forma...,"['Ph. D. in Economics', 'Prior college/univers...",Not Mentioned,241,"['Cover Letter', 'CV', 'Statement of Teaching ...",m
242,179557793,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",NaN,Open Until Filled,"['carry out their own research', 'participate ...","['strong academic record', 'trajectory to cont...","['doctorate in May 2025 or later', 'engage wit...",Not Mentioned,242,"['CV', 'Research statement (max 1,500 words)',...",n
243,179561620,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.287,TTU Post Doctoral Research Associate,Texas Tech University,"Lubbock, TX",Not mentioned,Not mentioned,['Performs specialized Post Doctoral work in t.

In [28]:
df = df.drop(columns=['idx'])
df

,id,text,link,date_posted,job_title,university,location,review_start,deadline,responsibilities,preferred_qualifications,required_skills,visa_sponsorship,documents,status
0,179507024,Assistant Professor of Economics\n============...,https://marylandconnect.wd1.myworkdayjobs.com/...,2026-07-22 17:45:52.497,Assistant Professor of Economics,Salisbury University,"Salisbury, MD",2026-09-11,2026-09-11,"['Undergraduate Teaching', 'Research', 'Public...",['Evidence of teaching effectiveness at the un...,"['Ph.D. in Economics', 'Demonstrated potential...",Not eligible,"['Curriculum Vitae', 'Cover Letter', 'Job Mark...",...
1,179568871,"Postdoctoral Fellow in Economic Science, Fall ...",https://apply.interfolio.com/194065,2026-09-25 17:35:19.32,"Postdoctoral Fellow in Economic Science, Fall ...",Chapman University,"Orange, CA",2027-08-01,2027-08-01,"['Design, program and run experiments using hu...",[],"['PhD in Economics, any field, with an emphasi...",Not eligible,"['Statement of interest', 'CV', 'Job Market Pa...",n
2,179582487,Visiting Assistant Professor of Global Studies...,https://apply.interfolio.com/195065,2026-10-08 11:32:45.427,Visiting Assistant Professor of Global Studies...,Colby College,"Waterville, ME",2026-12-04,2026-12-31,"['Teaching', 'Research', 'Service']","['Experience in critical political economy', '...",['Strong teaching excellence'],Not eligible,"['Resume', 'Cover Letter', 'Curriculum Vitae',...",n
3,179521072,Lecturer in Public and International Affairs\n...,https://www.click2apply.net/kNe4BNFDn2X4OHE6c2QNR,2026-08-06 13:26:41.55,Lecturer in Public and International Affairs,Princeton University,"Princeton, NJ",2026-03-19,NaN,"['Undergraduate Teaching', 'Co-teaching in are...",[],['Advanced degree or comparable experience in ...,Not eligible,"['Curriculum Vitae', 'Cover Letter', 'Teaching...",n
4,177351010,HKU Professoriate Recruitment (Business and Ec...,https://secure.dc5.pageuppeople.com/apply/932/...,2020-11-06 12:00:11.74,"Assistant, Associate, or Full Professor in Eme...",University of Hong Kong (HKU),"Hong Kong, Hong Kong",NaN,NaN,"['Undergraduate Teaching', 'Research', 'Public...",['Experience with relevant technologies and me...,"['Research in emerging academic disciplines', ...",Is eligible for H1B visa sponsorship,"['Resume', 'Cover Letter', 'Research Statement...",n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,179548639,Assistant Professor (Tenure Track) - Economics...,https://apply.interfolio.com/188852,2026-09-05 18:17:26.873,Assistant Professor (Tenure Track) - Economics,DePaul University,"Chicago, IL",2027-07-01,2027-09-11,"[""Teach undergraduate and graduate courses in ...",[],['Ph.D. in Economics or closely related field ...,Not Mentioned,"['Cover Letter', 'C.V.', 'Research Statement',...",y
241,179547688,Lecturer or Senior Lecturer in Economics\n====...,https://ukjobs.uky.edu/postings/647459,2026-09-04 08:54:25.533,Lecturer or Senior Lecturer in Economics,University of Kentucky,"Lexington, KY",2027-08-01,2027-08-01,"['Teaching in large-lecture sections', ""Teachi...",['Experience with teaching large lecture forma...,"['Ph. D. in Economics', 'Prior college/univers...",Not Mentioned,"['Cover Letter', 'CV', 'Statement of Teaching ...",m
242,179557793,SNF Agora Academy Postdoctoral Fellowship - 20...,https://apply.interfolio.com/193231,2026-09-15 21:16:25.827,SNF Agora Institute - Postdoctoral Fellowship,Johns Hopkins University,"Baltimore, MD",NaN,Open Until Filled,"['carry out their own research', 'participate ...","['strong academic record', 'trajectory to cont...","['doctorate in May 2025 or later', 'engage wit...",Not Mentioned,"['CV', 'Research statement (max 1,500 words)',...",n
243,179561620,TTU Post Doctoral Research Associate\n========...,https://apptrkr.com/9837874,2026-09-18 11:34:18.287,TTU Post Doctoral Research Associate,Texas Tech University,"Lubbock, TX",Not mentioned,Not mentioned,['Performs specialized Post Doctoral work in t...,['Demonstrated track record

In [29]:
df.to_csv(
    "data/hedjobs.csv",
    index=False,
    lineterminator="\r\n",  # Ensures standard carriage return + line feed for multi-line cells
    encoding="utf-8",
)

In [11]:
with open("data/html_cache/cv/0.html", "r", encoding="utf-8") as f:
    html_content = f.read()
soup = BeautifulSoup(html_content, "lxml")

In [18]:
soup.find('title').string.split('|')[-1].strip()

'38039753'

In [20]:
attribs = ['job-description', 'company-information']
for attrib_id in attribs:
    attrib_div = soup.find(id=attrib_id)
    if attrib_div:
        attrib = md.markdownify(str(attrib_div))
        print(attrib)

Job Details
-----------

**Assistant Professor of Economics**  

**Department of Economics**  
**College of Business and Economics**  
**Assistant Professor**  
**Fall 2027**

The Department of Economics at California State University,
Fullerton (CSUF) invites applications for two tenure-track
assistant professor positions in Economics with a focus on
empirical research. We are particularly interested in candidates
conducting research using machine learning methods.

**Equal Opportunity and Excellence in Education and
Employment**This position is open and available to all
regardless of race, sex, color, ethnicity or national origin.
Consistent with California law and federal civil rights laws,
California State University, Fullerton (CSUF) provides equal
opportunity in education and employment without unlawful
discrimination or preferential treatment based on race, sex, color,
ethnicity, or national origin. Our commitment to equal opportunity
means ensuring that every student and employ

In [23]:
apply_link = soup.find('a', attrs={'data-hook': 'apply-button'})

# Get the href URL
url = apply_link['href']
print('jobs.chronicle.com' + url)

jobs.chronicle.com/apply/38039753/assistant-professor-of-economics?LinkSource=JobDetails


In [24]:
pattern = r'"datePosted"\s*:\s*"([^"]+)"'
tag = soup.find('script', string=re.compile(pattern))
if tag:
    html_content = str(tag.string)
    match = re.search(pattern, html_content)
    if match:
        date_posted = match.group(1)
        print(date_posted)
print("Not found")

2026-09-16T04:00:00.000Z
Not found
